# Bielik - agent pogodowy z [PydanticAI](https://ai.pydantic.dev/)

[Bielik](https://bielik.ai/) to polski model językowy stworzony przez [SpeakLeash](https://speakleash.org/) i ICM. W tym notebooku zbudujemy agenta pogodowego (analogicznego do tego z `003-01. LLM-function_calling.ipynb`) opartego o model `SpeakLeash/bielik-11b-v3.0-instruct:bf16` hostowany lokalnie z użyciem [Ollama](https://ollama.com/).

[PydanticAI](https://ai.pydantic.dev/) to agent framework od zespołu Pydantica - mocno typowany, deklaratywny i z natywnym wsparciem dla wielu providerów (w tym Ollamy). Pozostałe trzy notebooki w tej serii pokazują tę samą funkcjonalność w innych podejściach:
- `010-01. Bielik-openai.ipynb` - manualna pętla na surowym SDK
- `010-03. Bielik-dspy.ipynb` - ReAct z DSPy
- `010-04. Bielik-llama-index.ipynb` - ReAct z LlamaIndex

## Function calling vs ReAct

PydanticAI **nie ma wbudowanego trybu ReAct opartego o parsowanie tekstu** - polega wyłącznie na strukturyzowanym _function calling_ (pole `tool_calls` w odpowiedzi modelu). Bielik domyślnie emituje wywołania funkcji jako tagi `<tool_call>{...}</tool_call>` w polu `content`, co Ollama bez customizacji wkleja jako zwykły tekst, zamiast strukturyzowane `tool_calls`.

Rozwiązanie: nadpisujemy template Bielika własnym Modelfile (`010-00. Bielik.Modelfile`), który deklaruje tagi `<tool_call>...</tool_call>` przylegle do zmiennej `{{ .ToolCalls }}` - dzięki temu parser Ollamy potrafi wyciągnąć strukturyzowane wywołania. Szczegóły co dokładnie zmieniamy względem oryginalnego Modelfile'a - patrz `010-00. Bielik.Modelfile.md`.

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile:
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
4. Działający serwis Ollama w tle (port `11434`).

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

In [ ]:
import requests
import json

## Funkcja pobierająca współrzędne geograficzne dla danej nazwy

In [ ]:
def get_geolocation(location):
    """
    Pobiera współrzędne geograficzne oraz dane lokalizacyjne.

    Parametry:
    location (str): Nazwa lokalizacji, dla której chcemy uzyskać współrzędne geograficzne.

    Zwraca:
    dict: Dane lokalizacyjne w formacie JSON.
    """
    print(f"[tool_call] get_geolocation(location={location!r})")

    geocode_endpoint = "https://nominatim.openstreetmap.org/search"
    geocode_params = {"q": location, "format": "json"}
    headers = {"User-Agent": "Python script"}

    geocode_response = requests.get(geocode_endpoint, params=geocode_params, headers=headers)
    geocode_data = geocode_response.json()

    simplified_data = {
        "name": geocode_data[0]["display_name"],
        "latitude": geocode_data[0]["lat"],
        "longitude": geocode_data[0]["lon"]
    }
    return simplified_data

geolocation_data = get_geolocation("Poznań")
print(json.dumps(geolocation_data, indent=4, ensure_ascii=False))

## Funkcja pobierająca informacje o pogodzie dla podanych współrzędnych geograficznych

In [ ]:
def get_wind_direction(degrees):
    """Konwertuje kierunek wiatru ze stopni na nazwy kierunków świata."""
    directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                  'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    index = int((degrees + 11.25) // 22.5) % 16
    return directions[index]

def get_current_weather(latitude, longitude):
    """
    Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych.

    Parametry:
    latitude (float): Szerokość geograficzna.
    longitude (float): Długość geograficzna.

    Zwraca:
    dict: Dane pogodowe w formacie JSON.
    """
    print(f"[tool_call] get_current_weather(latitude={latitude!r}, longitude={longitude!r})")

    weather_endpoint = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    weather_response = requests.get(weather_endpoint, params=weather_params)
    weather_data = weather_response.json()

    simplified_weather = {
        "temperature": f"{weather_data['current_weather']['temperature']} °C",
        "wind_speed": f"{weather_data['current_weather']['windspeed']} km/h",
        "wind_direction": get_wind_direction(weather_data['current_weather']['winddirection']),
        "is_day": "day" if weather_data['current_weather']['is_day'] else "night"
    }

    return simplified_weather

current_weather = get_current_weather(geolocation_data['latitude'], geolocation_data['longitude'])
print(json.dumps(current_weather, indent=4))

## Konfiguracja modelu

PydanticAI ma dedykowany `OllamaProvider`, który pod spodem podpina się do OpenAI-kompatybilnego endpointu Ollamy (`/v1/chat/completions`). Model owijamy w `OpenAIChatModel` - PydanticAI traktuje Ollamę jak każdego innego providera kompatybilnego z OpenAI Chat Completions API.

Używamy lokalnie zbudowanego modelu `bielik-tools` (z customowego Modelfile - patrz wymagania na górze).

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

# Provider wskazuje na OpenAI-kompatybilny endpoint Ollamy.
ollama_provider = OllamaProvider(base_url="http://host.docker.internal:11434/v1")

# Custom template w bielik-tools deklaruje tagi <tool_call>...</tool_call>
# przylegle do zmiennej {{ .ToolCalls }}, dzięki czemu parser Ollamy może
# wykryć prefix i zwrócić strukturyzowane tool_calls zamiast surowego tekstu.
bielik_pydantic_model = OpenAIChatModel(
    model_name="bielik-tools",
    provider=ollama_provider,
)

## Narzędzia i agent

PydanticAI generuje schemat JSON z **adnotacji typów** funkcji - dlatego nasze oryginalne `get_geolocation(location)` (bez adnotacji) trzeba opakować w cienkie wrappery z explicit typami. Tylko adnotacje są obowiązkowe; nazwa funkcji i docstring trafiają do opisu narzędzia.

Najpierw tworzymy `Agent` (bez listy narzędzi w konstruktorze), a potem rejestrujemy narzędzia **dekoratorem** `@pydantic_agent.tool_plain`. Wariant `.tool_plain` to wersja dla narzędzi, które nie potrzebują dostępu do kontekstu runu - jest też `@pydantic_agent.tool`, gdy funkcja musi dostać `RunContext` jako pierwszy argument.

In [ ]:
system_prompt = (
    "Jesteś pomocnym asystentem pogodowym. Jeśli do realizacji polecenia "
    "musisz ustalić współrzędne geograficzne jakiegoś miejsca, ZAWSZE "
    "pobierz je za pomocą narzędzia get_geolocation - nigdy nie podawaj "
    "współrzędnych z własnej wiedzy. Nie każde polecenie wymaga "
    "ustalania współrzędnych."
)

pydantic_agent = Agent(
    model=bielik_pydantic_model,
    system_prompt=system_prompt,
    retries=2, # liczba ponownych prób, jeśli model zwróci niepoprawny tool call
)

@pydantic_agent.tool_plain
def geolocation_tool_pa(location: str) -> dict:
    """Pobiera współrzędne geograficzne (latitude, longitude) dla nazwy miejscowości."""
    return get_geolocation(location)

@pydantic_agent.tool_plain
def current_weather_tool_pa(latitude: float, longitude: float) -> dict:
    """Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych."""
    return get_current_weather(latitude, longitude)

## Uruchomienie agenta

`agent.run(...)` jest async - używamy top-level `await`. Wynik to obiekt `AgentRunResult`, w którym `.output` zawiera ostateczną odpowiedź modelu, a `.all_messages()` pełną historię konwersacji (włącznie z tool callami i ich wynikami).

In [ ]:
result = await pydantic_agent.run("Opisz jaka jest pogoda w Poznaniu. Czy powinienem wychodzić na spacer w stroju plażowym i okularach przeciwsłonecznych?")
print("\n=== Ostateczna odpowiedź agenta ===")
print(result.output)

## Trajektoria agenta

`result.all_messages()` zwraca pełną listę wiadomości z runu - widać tu po kolei prompty, wywołania narzędzi (`ToolCallPart`) i ich wyniki (`ToolReturnPart`).

In [ ]:
for msg in result.all_messages():
    print(f"--- {type(msg).__name__} ---")
    for part in msg.parts:
        print(f"  {type(part).__name__}: {part}")